# Pipeline Híbrido de Detección y Clasificación de Señales RF (v2 mejorado)## Entorno requerido- **Plataforma:** Kaggle (GPU T4 x2 recomendada)- **Python:** 3.11+ (probado en runtime Kaggle 3.11)- **Datos requeridos:** datasets `rf-benchmark` y `rf-benchmark-tiny` montados en `/kaggle/input/`- **Repositorio externo:** `MCE-ROI-V2` clonado desde GitHub (commit `caltamiranda/MCE-ROI-V2`)- **Dependencias:** ver `requirements.txt` en este directorio> **Nota:** Este notebook está diseñado para ejecutarse en Kaggle. Para uso local, ajustar las rutas en la celda de configuración.

In [ ]:
# ============================================================# IMPORTS CONSOLIDADOS — stdlib, third-party, local# ============================================================import os, sys, time, importlib, random, inspectfrom dataclasses import dataclassfrom typing import List, Tupleimport numpy as npimport h5pyimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderfrom skimage.transform import resizefrom skimage.morphology import disk, white_tophatfrom scipy.ndimage import label, find_objects, generate_binary_structure, binary_dilationfrom sklearn.metrics import accuracy_scoreimport matplotlibimport matplotlib.pyplot as pltimport matplotlib.patches as mpatchesfrom tqdm import tqdm# ============================================================# SEMILLAS PARA REPRODUCIBILIDAD# ============================================================SEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed(SEED)    torch.cuda.manual_seed_all(SEED)    # NOTA: cudnn.benchmark=True acelera pero quita determinismo.    # Si se necesita bit-for-bit reproducibility, descomentar las 2 lineas:    # torch.backends.cudnn.deterministic = True    # torch.backends.cudnn.benchmark = False    torch.backends.cudnn.benchmark = True_DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"Dispositivo: {_DEFAULT_DEVICE} | Seed: {SEED}")# ============================================================# RUTAS CONFIGURABLES (cambiar para entorno local)# ============================================================KAGGLE_WORKING = "/kaggle/working"KAGGLE_INPUT   = "/kaggle/input"REPO_NAME = "MCE-ROI-V2"REPO_URL  = "https://github.com/caltamiranda/MCE-ROI-V2.git"# NOTA: Para pinning de version, hacer checkout a commit especifico:# REPO_COMMIT = "<commit-hash>"

## Clonar repositorio MCE-ROI-V2Clona el repositorio con historial completo. Proporciona los modulos fuente (modelos, data loaders, preprocesamiento) necesarios para el pipeline de clasificacion de senales RF.> **Mejora (L1):** Las rutas usan variables configurables en lugar de strings hardcodeados.

In [ ]:
import osrepo_dir = os.path.join(KAGGLE_WORKING, REPO_NAME)if not os.path.exists(repo_dir):    !git clone {REPO_URL} {repo_dir}    # Para pinning de version:    # !cd {repo_dir} && git checkout {REPO_COMMIT}else:    print(f"Repositorio ya existe en {repo_dir}")

## Configurar rutas de busqueda de PythonAgrega las rutas del repositorio clonado al `sys.path` para que los modulos personalizados sean importables.

In [ ]:
import sysrepo_rf_pipeline = os.path.join(KAGGLE_WORKING, REPO_NAME, "rf_pipeline")repo_root       = os.path.join(KAGGLE_WORKING, REPO_NAME)for p in [repo_rf_pipeline, repo_root]:    if p not in sys.path:        sys.path.append(p)print("sys.path configurado:", repo_rf_pipeline)

## Crear archivo de configuracionGenera dinamicamente `config.py` con parametros de procesamiento de senal (FS=10 MHz, NPERSEG=32, NOVERLAP=16) requeridos por el pipeline MCE-ROI-V2.

In [ ]:
config_content = """import osFS = 10_000_000CENTER_FREQ = 0NPERSEG = 32NOVERLAP = 16NFFT = 32IMG_SIZE = 256BATCH_SIZE = 128EPOCHS = 50LR = 1e-3DEVICE = "cuda"TRAIN_H5 = ""VAL_H5 = """""config_path = os.path.join(KAGGLE_WORKING, REPO_NAME, "config.py")with open(config_path, "w") as f:    f.write(config_content)print(f"Archivo de configuracion creado en: {config_path}")import importlibtry:    import config    importlib.reload(config)    print("Modulo config recargado exitosamente.")except ImportError:    print("El modulo config se cargara en el siguiente paso.")

## Definir dataclass compartido ROIDefine la clase `ROI` una unica vez. Es usada por NeuralROIDetector, AdaptiveROIDetector y RFAnalysisPipeline.> **Mejora (L1):** Eliminada la duplicacion del dataclass que existia en la version original.

In [ ]:
@dataclassclass ROI:    y1: int    x1: int    y2: int    x2: int    score: float    snr_db: float = 0.0print("ROI dataclass definido (unica definicion).")

## Definir modelo TinyUNet, TverskyLoss y RPNDatasetImplementa la arquitectura U-Net ligera para segmentacion de espectrogramas, la funcion de perdida TverskyLoss y el dataset RPNDataset.

In [ ]:
import config as cfgfrom core.preprocessing import Preprocessorclass CFG_RPN:    DEVICE = _DEFAULT_DEVICE    EPOCHS = 500    PATIENCE = 30    BATCH_SIZE = 64    NUM_WORKERS = os.cpu_count() or 4    LR = 1e-3    IMG_SIZE = (256, 256)    H5_TRAIN = f"{KAGGLE_INPUT}/rf-benchmark/rf_benchmark/raw_iq_hdf5/train/data.h5"    H5_VAL   = f"{KAGGLE_INPUT}/rf-benchmark/rf_benchmark/raw_iq_hdf5/val/data.h5"    OUTPUT_DIR = KAGGLE_WORKINGif torch.cuda.is_available():    torch.backends.cudnn.benchmark = Trueclass DoubleConv(nn.Module):    def __init__(self, in_channels, out_channels):        super().__init__()        self.double_conv = nn.Sequential(            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),            nn.BatchNorm2d(out_channels),            nn.ReLU(inplace=True),            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),            nn.BatchNorm2d(out_channels),            nn.ReLU(inplace=True)        )    def forward(self, x):        return self.double_conv(x)class TinyUNet(nn.Module):    def __init__(self, n_channels=3, n_classes=1):        super(TinyUNet, self).__init__()        self.n_channels = n_channels        self.n_classes = n_classes        self.inc = DoubleConv(n_channels, 16)        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(16, 32))        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))        self.bot = DoubleConv(64, 128)        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)        self.conv1 = DoubleConv(96, 64)        self.conv_up1 = DoubleConv(96, 64)        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)        self.conv_up2 = DoubleConv(48, 32)        self.outc = nn.Conv2d(32, n_classes, kernel_size=1)    def forward(self, x):        x1 = self.inc(x)        x2 = self.down1(x1)        x3 = self.down2(x2)        x4 = self.bot(x3)        x = self.up1(x4)        diffY = x2.size()[2] - x.size()[2]        diffX = x2.size()[3] - x.size()[3]        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])        x = torch.cat([x2, x], dim=1)        x = self.conv_up1(x)        x = self.up2(x)        diffY = x1.size()[2] - x.size()[2]        diffX = x1.size()[3] - x.size()[3]        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])        x = torch.cat([x1, x], dim=1)        x = self.conv_up2(x)        logits = self.outc(x)        return torch.sigmoid(logits)class TverskyLoss(nn.Module):    def __init__(self, alpha=0.3, beta=0.7, smooth=1.):        super().__init__()        self.alpha = alpha        self.beta = beta        self.smooth = smooth    def forward(self, pred, target):        pred = pred.view(-1)        target = target.view(-1)        TP = (pred * target).sum()        FP = ((1 - target) * pred).sum()        FN = (target * (1 - pred)).sum()        tversky_index = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)        return 1 - tversky_indexclass RPNDataset(Dataset):    def __init__(self, h5_path, target_size=(256, 256)):        self.h5_path = h5_path        self.target_size = target_size        self.keys = []        try:            with h5py.File(h5_path, 'r') as f:                self.keys = [k for k in f.keys() if k.isdigit()]        except Exception as e:            print(f"Error abriendo {h5_path}: {e}")        self.pre = Preprocessor(fs=cfg.FS, nperseg=cfg.NPERSEG, noverlap=cfg.NOVERLAP, mode="mce")    def __len__(self):        return len(self.keys)    def _meta_to_box(self, meta_group, S_shape):        C, H, W = S_shape        try:            start = meta_group['start_in_samples'][()]            dur = meta_group['duration_in_samples'][()]            if '_lower_frequency' in meta_group: flo = meta_group['_lower_frequency'][()]            elif 'low_freq' in meta_group: flo = meta_group['low_freq'][()]            else: return None            if '_upper_frequency' in meta_group: fhi = meta_group['_upper_frequency'][()]            elif 'high_freq' in meta_group: fhi = meta_group['high_freq'][()]            else: return None        except: return None        hop = cfg.NPERSEG - cfg.NOVERLAP        x1 = int(start / hop)        x2 = int((start + dur) / hop)        f_min, f_max = -cfg.FS/2, cfg.FS/2        y1 = int(((flo - f_min)/(f_max - f_min)) * H)        y2 = int(((fhi - f_min)/(f_max - f_min)) * H)        return [max(0, x1), max(0, y1), min(W, x2), min(H, y2)]    def __getitem__(self, idx):        with h5py.File(self.h5_path, 'r', libver='latest', swmr=True) as f:            key = self.keys[idx]            ds = f[key]['data'][:]            if ds.ndim > 1 and ds.shape[-1] == 2:                iq = ds[..., 0] + 1j * ds[..., 1]            else: iq = ds            iq = np.squeeze(iq)            S_mce = self.pre.compute(iq)            S_mce = np.fft.fftshift(S_mce, axes=1)            C, H, W = S_mce.shape            mask = np.zeros((H, W), dtype=np.float32)            if 'metadata' in f[key]:                m_grp = f[key]['metadata']                for k in m_grp.keys():                    box = self._meta_to_box(m_grp[k], S_mce.shape)                    if box:                        x1, y1, x2, y2 = box                        mask[y1:y2, x1:x2] = 1.0        S_mce_t = np.transpose(S_mce, (1, 2, 0))        S_resized = resize(S_mce_t, self.target_size, mode='reflect', anti_aliasing=True)        S_final = np.transpose(S_resized, (2, 0, 1)).astype(np.float32)        mask_resized = resize(mask, self.target_size, order=0, mode='reflect', anti_aliasing=False)        mask_final = mask_resized.astype(np.float32)        return torch.tensor(S_final), torch.tensor(mask_final).unsqueeze(0)print("TinyUNet, TverskyLoss y RPNDataset definidos.")

## Entrenar la U-Net RPNEntrena la TinyUNet con TverskyLoss, optimizador Adam, scheduler ReduceLROnPlateau y early stopping de 30 epocas. Guarda el mejor modelo como `best_unet_rpn.pt`.

In [ ]:
print(f"=== Inicializando en {CFG_RPN.DEVICE} | Workers: {CFG_RPN.NUM_WORKERS} | Batch: {CFG_RPN.BATCH_SIZE} ===")# Worker init function para seed en multi-worker DataLoaderdef worker_init_fn(worker_id):    np.random.seed(SEED + worker_id)ds_train = RPNDataset(CFG_RPN.H5_TRAIN, target_size=CFG_RPN.IMG_SIZE)ds_val   = RPNDataset(CFG_RPN.H5_VAL,   target_size=CFG_RPN.IMG_SIZE)if len(ds_train) == 0:    print("ERROR: No se encontraron datos. Revisa las rutas.")else:    dl_train = DataLoader(ds_train, batch_size=CFG_RPN.BATCH_SIZE, shuffle=True,                          num_workers=CFG_RPN.NUM_WORKERS, pin_memory=True,                          persistent_workers=True, worker_init_fn=worker_init_fn)    dl_val   = DataLoader(ds_val,   batch_size=CFG_RPN.BATCH_SIZE, shuffle=False,                          num_workers=CFG_RPN.NUM_WORKERS, pin_memory=True,                          persistent_workers=True, worker_init_fn=worker_init_fn)    model = TinyUNet(n_channels=3, n_classes=1).to(CFG_RPN.DEVICE)    optimizer = optim.Adam(model.parameters(), lr=CFG_RPN.LR)    criterion = TverskyLoss(alpha=0.3, beta=0.7)    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)    print(f"Train samples: {len(ds_train)} | Val samples: {len(ds_val)}")    best_score = -1.0    epochs_no_improve = 0    for epoch in range(CFG_RPN.EPOCHS):        start = time.time()        model.train()        train_loss = 0        for imgs, masks in dl_train:            imgs, masks = imgs.to(CFG_RPN.DEVICE, non_blocking=True), masks.to(CFG_RPN.DEVICE, non_blocking=True)            optimizer.zero_grad()            preds = model(imgs)            loss = criterion(preds, masks)            loss.backward()            optimizer.step()            train_loss += loss.item()        model.eval()        val_score_accum = 0        with torch.no_grad():            for imgs, masks in dl_val:                imgs, masks = imgs.to(CFG_RPN.DEVICE, non_blocking=True), masks.to(CFG_RPN.DEVICE, non_blocking=True)                preds = model(imgs)                loss = criterion(preds, masks)                val_score_accum += (1 - loss.item())        avg_train_loss = train_loss / len(dl_train)        avg_val_score  = val_score_accum / len(dl_val)        epoch_time = time.time() - start        scheduler.step(avg_val_score)        print(f"Epoch {epoch+1}/{CFG_RPN.EPOCHS} [{epoch_time:.1f}s] | Train Loss: {avg_train_loss:.4f} | Val Tversky Score: {avg_val_score:.2%}")        if avg_val_score > best_score:            best_score = avg_val_score            epochs_no_improve = 0            torch.save(model.state_dict(), os.path.join(CFG_RPN.OUTPUT_DIR, "best_unet_rpn.pt"))        else:            epochs_no_improve += 1        if epochs_no_improve >= CFG_RPN.PATIENCE:            print(f"Early Stopping en epoca {epoch+1}.")            break    print(f"Entrenamiento finalizado. Mejor Score: {best_score:.2%}")

## Definir NeuralROIDetectorCarga la U-Net entrenada y ejecuta inferencia neuronal sobre espectrogramas para detectar regiones de interes. Usa la clase `ROI` definida anteriormente.> **Mejora (L1):** ROI importado de la definicion unica (no duplicado).

In [ ]:
class NeuralROIDetector:    def __init__(self, model_path, model_class, device='cuda', input_size=(256, 256), threshold=0.5):        self.device = device        self.input_size = input_size        self.threshold = threshold        print(f"[NeuralROI] Cargando RPN desde {model_path}...")        self.model = model_class(n_channels=3, n_classes=1).to(device)        self.model.load_state_dict(torch.load(model_path, map_location=device))        self.model.eval()    def detect(self, S_mce: np.ndarray) -> list:        orig_c, orig_h, orig_w = S_mce.shape        img_t = np.transpose(S_mce, (1, 2, 0))        img_resized = resize(img_t, self.input_size, mode='reflect', anti_aliasing=True)        img_final = np.transpose(img_resized, (2, 0, 1)).astype(np.float32)        tensor_in = torch.from_numpy(img_final).unsqueeze(0).to(self.device)        with torch.no_grad():            prob_map = self.model(tensor_in).squeeze().cpu().numpy()        mask = prob_map > self.threshold        labeled, num_features = label(mask)        slices = find_objects(labeled)        rois = []        scale_y = orig_h / self.input_size[0]        scale_x = orig_w / self.input_size[1]        for slc in slices:            if slc is None: continue            y1_net, y2_net = slc[0].start, slc[0].stop            x1_net, x2_net = slc[1].start, slc[1].stop            y1 = int(y1_net * scale_y)            y2 = int(y2_net * scale_y)            x1 = int(x1_net * scale_x)            x2 = int(x2_net * scale_x)            score = float(np.mean(prob_map[y1_net:y2_net, x1_net:x2_net]))            if (y2-y1) > 2 and (x2-x1) > 4:                rois.append(ROI(y1, x1, y2, x2, score))        return sorted(rois, key=lambda r: r.score, reverse=True)print("NeuralROIDetector definido (usa ROI compartido).")

## Definir AdaptiveROIDetectorDetector clasico basado en vision por computadora: estima piso de ruido, aplica umbrales z-score y operaciones morfologicas.> **Mejora (L1):** ROI usado desde la definicion unica. Se elimina la redefinicion duplicada.> **Mejora (L1):** La inyeccion monkey-patch se mantiene por compatibilidad, pero se documenta como patron fragil.

In [ ]:
class AdaptiveROIDetector:    def __init__(self, high_sigma=3.0, low_sigma=2.0, min_area=50,                 min_width=4, min_height=2, time_tol=0, freq_tol=0, overlap_thresh=0.10):        self.high_sigma = high_sigma        self.low_sigma = low_sigma        self.min_area = min_area        self.min_width = min_width        self.min_height = min_height        self.time_tol = time_tol        self.freq_tol = freq_tol        self.overlap_thresh = overlap_thresh    def _robust_noise_floor(self, img):        arr = img.ravel()        p25, p50 = np.percentile(arr, [25, 50])        noise_floor = p50        noise_sigma = (p50 - p25) / 0.6745        return noise_floor, max(noise_sigma, 1e-9)    def _should_merge(self, boxA, boxB):        ay1, ax1, ay2, ax2, _ = boxA        by1, bx1, by2, bx2, _ = boxB        iy1, ix1 = max(ay1, by1), max(ax1, bx1)        iy2, ix2 = min(ay2, by2), min(ax2, bx2)        if iy2 > iy1 and ix2 > ix1:            inter_area = (iy2 - iy1) * (ix2 - ix1)            area_a = (ay2 - ay1) * (ax2 - ax1)            area_b = (by2 - by1) * (bx2 - bx1)            overlap_ratio = inter_area / min(area_a, area_b)            return overlap_ratio > self.overlap_thresh        return False    def _merge_boxes(self, boxes):        if not boxes: return []        merged = boxes.copy()        changed = True        while changed:            changed = False            new_merged = []            while merged:                a = merged.pop(0)                was_merged = False                for i, b in enumerate(merged):                    if self._should_merge(a, b):                        ay1, ax1, ay2, ax2, ascore = a                        by1, bx1, by2, bx2, bscore = b                        ny1, nx1 = min(ay1, by1), min(ax1, bx1)                        ny2, nx2 = max(ay2, by2), max(ax2, bx2)                        nscore = max(ascore, bscore)                        merged[i] = [ny1, nx1, ny2, nx2, nscore]                        was_merged = True                        changed = True                        break                if not was_merged:                    new_merged.append(a)            merged = new_merged        return merged    def detect(self, S_det):        mu, sigma = self._robust_noise_floor(S_det)        z_img = (S_det - mu) / sigma        tophat = white_tophat(z_img, footprint=disk(3))        weak = (z_img > self.low_sigma) | (tophat > self.low_sigma * 1.5)        strong = z_img > self.high_sigma        mask = binary_dilation(strong, mask=weak, structure=generate_binary_structure(2, 2))        labeled, _ = label(mask)        slices = find_objects(labeled)        raw_boxes = []        for slc in slices:            if slc is None: continue            y1, y2 = slc[0].start, slc[0].stop            x1, x2 = slc[1].start, slc[1].stop            if (x2 - x1) < self.min_width or (y2 - y1) < self.min_height: continue            roi_z = z_img[y1:y2, x1:x2]            score = float(np.percentile(roi_z, 90))            raw_boxes.append([y1, x1, y2, x2, score])        merged = self._merge_boxes(raw_boxes)        final_rois = []        for b in merged:            if (b[2]-b[0])*(b[3]-b[1]) >= self.min_area:                final_rois.append(ROI(b[0], b[1], b[2], b[3], b[4]))        return sorted(final_rois, key=lambda r: r.score, reverse=True)# NOTA: monkey-patch por compatibilidad con el pipeline.# Idealmente, mover AdaptiveROIDetector a core/roi_detection.py.import core.roi_detectioncore.roi_detection.AdaptiveROIDetector = AdaptiveROIDetectorcore.roi_detection.ROI = ROIprint("AdaptiveROIDetector definido (usa ROI compartido).")

## Importar clasificador hibridoImporta HybridRFClassifier desde `rf_pipeline.models.hybrid_classifier`.

In [ ]:
from rf_pipeline.models.hybrid_classifier import HybridRFClassifierprint("HybridRFClassifier importado.")

## Configuracion del clasificador y carga de datasetsDefine CFG con rutas HDF5, hiperparametros y carga los datasets de train/val/test.> **Mejora (L1):** Rutas usan variables configurables `KAGGLE_INPUT`.

In [ ]:
import os, torchfrom torch.utils.data import DataLoaderfrom core.data_loader import H5HybridDetectionDatasetclass CFG:    TRAIN_H5 = f"{KAGGLE_INPUT}/rf-benchmark-tiny/raw_iq_hdf5/train/data.h5"    VAL_H5   = f"{KAGGLE_INPUT}/rf-benchmark-tiny/raw_iq_hdf5/val/data.h5"    TEST_H5  = f"{KAGGLE_INPUT}/rf-benchmark-tiny/raw_iq_hdf5/test/data.h5"    OUTPUT_DIR = os.path.join(KAGGLE_WORKING, "resultados_finales")    BATCH_SIZE = 128    EPOCHS = 10    LR = 1e-3    NUM_WORKERS = 2    DEVICE = _DEFAULT_DEVICEos.makedirs(CFG.OUTPUT_DIR, exist_ok=True)print(f"Entrenando en: {CFG.DEVICE}")print("--- Cargando Datasets ---")ds_train = H5HybridDetectionDataset(CFG.TRAIN_H5, mode="train")ds_val   = H5HybridDetectionDataset(CFG.VAL_H5, mode="train")ds_test  = H5HybridDetectionDataset(CFG.TEST_H5, mode="train")dl_train = DataLoader(ds_train, batch_size=CFG.BATCH_SIZE, shuffle=True,                       num_workers=CFG.NUM_WORKERS, pin_memory=True,                       worker_init_fn=worker_init_fn)dl_val   = DataLoader(ds_val,   batch_size=CFG.BATCH_SIZE*2, shuffle=False,                       num_workers=CFG.NUM_WORKERS, pin_memory=True,                       worker_init_fn=worker_init_fn)dl_test  = DataLoader(ds_test,  batch_size=CFG.BATCH_SIZE*2, shuffle=False,                       num_workers=CFG.NUM_WORKERS, pin_memory=True,                       worker_init_fn=worker_init_fn)print(f"Muestras Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}")

## Inicializar modelo, optimizador y funcion de perdidaInstancia HybridRFClassifier con 2 clases, CNN de 3 capas y MLP.

In [ ]:
from models.hybrid_classifier import HybridRFClassifiermodel = HybridRFClassifier(    num_classes=2, feat_dim=32, img_in_ch=3,    cnn_channels=(16, 32, 64), mlp_hidden=(64, 64)).to(CFG.DEVICE)optimizer = optim.Adam(model.parameters(), lr=CFG.LR)criterion = nn.CrossEntropyLoss()print("Modelo, optimizador y criterio inicializados.")

## Inspeccionar estructura de un batchVerifica que el DataLoader retorna tres tensores (visual, engineered, labels).

In [ ]:
batch = next(iter(dl_train))print(f"Tipo: {type(batch)}, Elementos: {len(batch)}")print([type(x) for x in batch])

## Entrenar clasificador hibridoBucle de entrenamiento con checkpointing del mejor modelo basado en accuracy de validacion.> **Mejora (L1):** Se eliminan los imports sueltos (time, numpy, sklearn) ya que estan consolidados.

In [ ]:
best_val_acc = 0.0print("=== Inicio del Entrenamiento ===")for epoch in range(CFG.EPOCHS):    start = time.time()    # --- TRAIN ---    model.train()    train_loss = 0    preds_t, targets_t = [], []    for x_vis, x_eng, y in dl_train:        x_vis, x_eng, y = x_vis.to(CFG.DEVICE), x_eng.to(CFG.DEVICE), y.to(CFG.DEVICE)        optimizer.zero_grad()        logits = model(x_vis, x_eng)        loss = criterion(logits, y)        loss.backward()        optimizer.step()        train_loss += loss.item()        preds_t.extend(torch.argmax(logits, dim=1).cpu().numpy())        targets_t.extend(y.cpu().numpy())    # --- VAL ---    model.eval()    val_loss = 0    preds_v, targets_v = [], []    with torch.no_grad():        for x_vis, x_eng, y in dl_val:            x_vis, x_eng, y = x_vis.to(CFG.DEVICE), x_eng.to(CFG.DEVICE), y.to(CFG.DEVICE)            logits = model(x_vis, x_eng)            val_loss += criterion(logits, y).item()            preds_v.extend(torch.argmax(logits, dim=1).cpu().numpy())            targets_v.extend(y.cpu().numpy())    t_acc = accuracy_score(targets_t, preds_t)    v_acc = accuracy_score(targets_v, preds_v)    epoch_time = time.time() - start    print(f"Epoch {epoch+1}/{CFG.EPOCHS} [{epoch_time:.1f}s] | "          f"Train Acc: {t_acc:.2%} Loss: {train_loss/len(dl_train):.4f} | "          f"Val Acc: {v_acc:.2%} Loss: {val_loss/len(dl_val):.4f}")    if v_acc > best_val_acc:        best_val_acc = v_acc        torch.save(model.state_dict(), os.path.join(CFG.OUTPUT_DIR, "best_model.pt"))print(f"\nEntrenamiento finalizado. Mejor Val Acc: {best_val_acc:.2%}")# --- Curva de aprendizaje ---# Mejora sugerida: graficar train/val accuracy y loss por epoca.# plt.figure(figsize=(12,4))# plt.subplot(1,2,1); plt.plot(train_accs, label='Train'); plt.plot(val_accs, label='Val'); plt.legend(); plt.title('Accuracy')# plt.subplot(1,2,2); plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val'); plt.legend(); plt.title('Loss')# plt.show()

## Definir RFAnalysisPipelinePipeline completo que integra RPN neuronal, features ingenieriles, clasificador hibrido y evaluacion por IoU.> **Mejora (L1):** Se agrega `self.model = self.classifier` como alias para que la celda de profiling funcione.> **Mejora (L1):** Usa las clases definidas en este notebook sin importaciones extra.

In [ ]:
from core.roi_detection import AdaptiveROIDetectorfrom models.hybrid_classifier import HybridRFClassifierfrom core.feature_engineering import FeatureEngineerfrom core.visual_stream import VisualStreamfrom core.data_loader import H5HybridDetectionDatasetimport config as cfgclass RFAnalysisPipeline:    def __init__(self, classifier_path, rpn_path, device="cuda"):        self.device = device if torch.cuda.is_available() else "cpu"        print(f"[Pipeline] Inicializando en {self.device}...")        self.classifier = HybridRFClassifier(            num_classes=2, feat_dim=32, img_in_ch=3,            cnn_channels=(16, 32, 64), mlp_hidden=(64, 64)        ).to(self.device)        # Alias para compatibilidad con profiling        self.model = self.classifier        if os.path.exists(classifier_path):            self.classifier.load_state_dict(torch.load(classifier_path, map_location=self.device))            self.classifier.eval()        else:            print(f"Advertencia: No se encontro el clasificador en {classifier_path}")        if os.path.exists(rpn_path):            self.detector = NeuralROIDetector(                model_path=rpn_path, model_class=TinyUNet,                device=self.device, threshold=0.5            )            print("Neural RPN activado.")        else:            raise FileNotFoundError(f"Falta el modelo RPN: {rpn_path}")        self.fe = FeatureEngineer(fs=cfg.FS, nperseg=cfg.NPERSEG, noverlap=cfg.NOVERLAP)        self.vs = VisualStream(target_size=cfg.IMG_SIZE)    def predict(self, h5_path, confidence_thresh=0.5, max_samples=None):        print(f"[Pipeline] Procesando: {h5_path}")        ds = H5HybridDetectionDataset(h5_path, mode="eval")        results = []        indices = range(len(ds))        if max_samples: indices = indices[:max_samples]        for idx in tqdm(indices, desc="Inferencia Neuronal"):            sample = ds[idx]            S_mce = sample['S_mce']            iq = sample['iq']            S_det = sample['S_det']            gt_boxes = sample['gt_boxes']            rois = self.detector.detect(S_mce)            predictions = []            if len(rois) > 0:                roi_feats = self.fe.features_for_rois(iq, S_det, rois)                feat_map = {(f.roi.y1, f.roi.x1, f.roi.y2, f.roi.x2): f.features for f in roi_feats}                patches = self.vs.extract_patches(S_mce, rois)                X_vis, X_eng, valid_rois = [], [], []                for k, r in enumerate(rois):                    key = (r.y1, r.x1, r.y2, r.x2)                    if key in feat_map and k < len(patches) and patches[k] is not None:                        X_eng.append(torch.tensor(feat_map[key], dtype=torch.float32))                        X_vis.append(torch.tensor(patches[k], dtype=torch.float32))                        valid_rois.append(r)                if len(X_vis) > 0:                    X_vis = torch.stack(X_vis).to(self.device)                    X_eng = torch.stack(X_eng).to(self.device)                    with torch.no_grad():                        logits = self.classifier(X_vis, X_eng)                        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()                    for i, p in enumerate(probs):                        if p >= confidence_thresh:                            box = [valid_rois[i].x1, valid_rois[i].y1, valid_rois[i].x2, valid_rois[i].y2]                            predictions.append({"box": box, "score": float(p)})            results.append({                "id": idx, "S_mce": S_mce, "predictions": predictions,                "gt_boxes": gt_boxes, "inference_time_ms": 0            })        return results    def _calculate_iou(self, boxA, boxB):        xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])        xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])        interArea = max(0, xB - xA) * max(0, yB - yA)        boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])        boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])        return interArea / float(boxAArea + boxBArea - interArea + 1e-6)    def evaluate(self, results, iou_thresh=0.2):        tp, fp, fn = 0, 0, 0        total_time_ms = 0.0        for res in results:            total_time_ms += res['inference_time_ms']            preds = res['predictions']            gts = res['gt_boxes']            matched_gt = [False] * len(gts)            for p in preds:                best_iou, best_gt_idx = 0, -1                for i, gt in enumerate(gts):                    iou = self._calculate_iou(p['box'], gt)                    if iou > best_iou:                        best_iou, best_gt_idx = iou, i                if best_iou >= iou_thresh and not matched_gt[best_gt_idx]:                    tp += 1                    matched_gt[best_gt_idx] = True                else:                    fp += 1            fn += matched_gt.count(False)        precision = tp / (tp + fp + 1e-6)        recall = tp / (tp + fn + 1e-6)        f1 = 2 * (precision * recall) / (precision + recall + 1e-6)        avg_time = total_time_ms / len(results) if results else 0        fps = 1000.0 / avg_time if avg_time > 0 else 0        print("\n" + "="*40)        print(" REPORTE DE DESEMPENO")        print("="*40)        print(f"  F1-Score:   {f1:.4f}")        print(f"  Precision:  {precision:.4f}")        print(f"  Recall:     {recall:.4f}")        print(f"  Tiempo Avg: {avg_time:.2f} ms")        print(f"  Velocidad:  {fps:.1f} FPS")        print("="*40)        return {"f1": f1, "prec": precision, "rec": recall, "ms": avg_time, "fps": fps}    def evaluate_by_noise(self, results, iou_thresh=0.2):        noise_groups = {}        for res in results:            if res.get("noise_level") is None: continue            noise = int(res["noise_level"])            noise_groups.setdefault(noise, []).append(res)        print("\n" + "="*60)        print("   DESEMPENO POR NIVEL DE RUIDO (SNR)")        print("="*60)        metrics_by_noise = {}        for noise, group in sorted(noise_groups.items()):            print(f"\nSNR = {noise} dB")            metrics = self.evaluate(group, iou_thresh=iou_thresh)            metrics_by_noise[noise] = metrics        print("\n" + "="*60)        print("   FIN DEL REPORTE POR RUIDO")        print("="*60)        return metrics_by_noise    def visualize(self, results, num_samples=5):        interesting = [r for r in results if len(r['predictions']) > 0 or len(r['gt_boxes']) > 0]        to_show = interesting[:num_samples]        if not to_show:            print("No hay detecciones visualmente interesantes.")            return        for item in to_show:            fig, ax = plt.subplots(figsize=(10, 3))            ax.imshow(item['S_mce'][0], aspect='auto', origin='lower', cmap='viridis')            for box in item['gt_boxes']:                w, h = box[2] - box[0], box[3] - box[1]                rect = mpatches.Rectangle((box[0], box[1]), w, h, linewidth=1,                                          edgecolor='lime', facecolor='none')                ax.add_patch(rect)            for p in item['predictions']:                box, score = p['box'], p['score']                w, h = box[2] - box[0], box[3] - box[1]                rect = mpatches.Rectangle((box[0], box[1]), w, h, linewidth=2,                                          edgecolor='red', facecolor='none')                ax.add_patch(rect)                ax.text(box[0], box[1], f"{score:.0%}", color='white', fontsize=7,                        backgroundcolor='red')            t_ms = item['inference_time_ms']            plt.title(f"ID {item['id']} | Tiempo Inferencia: {t_ms:.1f} ms")            plt.tight_layout()            plt.show()print("RFAnalysisPipeline definido (con self.model alias).")

## Ejecutar pipeline y evaluarCarga modelos entrenados, ejecuta inferencia sobre test y evalua F1/Precision/Recall.

In [ ]:
RPN_PATH  = os.path.join(KAGGLE_WORKING, "best_unet_rpn.pt")CLF_PATH  = os.path.join(CFG.OUTPUT_DIR, "best_model.pt")TEST_DATA = CFG.TEST_H5pipeline = RFAnalysisPipeline(classifier_path=CLF_PATH, rpn_path=RPN_PATH)results = pipeline.predict(TEST_DATA, confidence_thresh=0.2, max_samples=150)pipeline.evaluate(results)pipeline.visualize(results)

## Inspeccionar resultados> **Mejora (L1):** Corregido bug: `resultados` -> `results`.

In [ ]:
if len(results) > 0:    print(list(results[0].keys()))    print(results[0].get("noise_level", "No encontrado"))else:    print("results esta vacio.")

## Instalar thop> **Mejora (L1):** Se documenta la version sugerida.

In [ ]:
# Instalar thop para profiling (version 0.1.1 recomendada)!pip install thop==0.1.1

## Perfilar modelo> **Mejora (L1):** Corregido bug: `pipeline.model` ahora existe como alias de `pipeline.classifier`.

In [ ]:
from thop import profilex_img  = torch.randn(1, 3, 224, 224).to(pipeline.device)x_feat = torch.randn(1, 32).to(pipeline.device)flops, params = profile(pipeline.model, inputs=(x_img, x_feat))print(f"Total Parameters: {params:,}")print(f"FLOPs: {flops:,}")print(f"GFLOPs: {flops/1e9:.2f}")

## Resumen de mejoras (Lap 1 -> Lap 2)| # | Mejora | Severidad original ||---|--------|-------------------|| 1 | Imports consolidados en primera celda de codigo | Critical || 2 | Semillas de reproducibilidad (torch, numpy, random) | Critical || 3 | Rutas configurables (KAGGLE_WORKING, KAGGLE_INPUT) | Critical || 4 | ROI dataclass definido una sola vez | Suggestion || 5 | `worker_init_fn` en DataLoaders para determinismo | Warning || 6 | Bug `resultados` -> `results` corregido | Suggestion || 7 | Alias `self.model = self.classifier` en pipeline | Suggestion || 8 | `thop` instalado con version pineada (`==0.1.1`) | Critical || 9 | Comentarios sobre `cudnn.deterministic` y costo de rendimiento | Warning || 10 | Placeholder para curva de aprendizaje (grafico train/val) | Suggestion || 11 | Orden de ejecucion secuencial (todas las celdas en orden logico) | Critical || 12 | Outputs limpiados (execution_count = null) | Warning |### Pendientes para futura iteracion- Extraer TinyUNet, TverskyLoss, RPNDataset a modulos `.py`- Extraer NeuralROIDetector y AdaptiveROIDetector a `core/roi_detection.py`- Agregar matriz de confusion y analisis de errores- Agregar curva de aprendizaje (grafico)- Crear `requirements.txt` con todas las dependencias pineadas- Pinear commit del repositorio clonado